# 01. PDF 살펴보기

RAG를 만들기 전에 보고서 PDF 3개의 상태를 직접 확인합니다.

**이 노트북이 답할 질문**
1. 각 문서에 표와 차트가 몇 개 있나?
2. 글자가 올바른 순서로 추출되나?
3. D02, D03은 몇 쪽부터 몇 쪽까지 쓸까?
4. 차트를 어떤 방식으로 꺼내야 하나?

- API 키가 필요 없습니다. (PyMuPDF만 사용, 비용 0원)
- 쪽 그림은 노트북에 띄우지 않고 `data/explore/`에 파일로 저장합니다. (원본 재배포 방지, GitHub에 안 올라감)
- 문서 정보: `docs/data_sources.md`

## ① 준비
라이브러리를 불러오고 PDF 폴더 경로를 정합니다. 3개 파일이 모두 "있음"으로 나오는지 확인합니다.

In [14]:
from pathlib import Path
import re

import pandas as pd
import pymupdf

# 노트북은 notebooks/ 폴더에서 실행되므로, 한 단계 위가 프로젝트 폴더
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PDF_DIR = PROJECT_DIR / "data" / "pdf"
OUT_DIR = PROJECT_DIR / "data" / "explore"  # 쪽 그림 저장 위치 (GitHub에 안 올라감)
OUT_DIR.mkdir(parents=True, exist_ok=True)

DOCS = {
    "D01": "2026_youth_employment_survey.pdf",
    "D02": "2025_content_industry_trend.pdf",
    "D03": "2025_social_survey.pdf",
}

for doc_id, file_name in DOCS.items():
    print(doc_id, file_name, "있음" if (PDF_DIR / file_name).exists() else "없음")

D01 2026_youth_employment_survey.pdf 있음
D02 2025_content_industry_trend.pdf 있음
D03 2025_social_survey.pdf 있음


## ② 기본 정보
문서별 쪽수, 파일 크기, 암호 여부를 확인합니다.

In [15]:
docs = {doc_id: pymupdf.open(PDF_DIR / file_name) for doc_id, file_name in DOCS.items()}

info_rows = []
for doc_id, doc in docs.items():
    info_rows.append({
        "문서": doc_id,
        "쪽수": doc.page_count,
        "크기(MB)": round((PDF_DIR / DOCS[doc_id]).stat().st_size / 1_000_000, 1),
        "암호": doc.is_encrypted,
    })
pd.DataFrame(info_rows)

,문서,쪽수,크기(MB),암호
0,D01,40,9.5,False
1,D02,282,22.3,False
2,D03,133,3.2,False


## ③ 쪽별 글자 수
쪽마다 글자 수와 숫자 비율을 표로 정리합니다.
- 글자 수가 거의 없는 쪽 → 표지, 간지, 빈 쪽
- 숫자 비율이 높은 쪽 → 통계표일 가능성

In [16]:
def digit_ratio(text):
    """공백을 뺀 글자 중 숫자의 비율. 높으면 통계표 쪽일 가능성이 큼"""
    chars = re.sub(r"\s", "", text)
    if not chars:
        return 0.0
    return round(sum(c.isdigit() for c in chars) / len(chars), 2)

page_rows = []
for doc_id, doc in docs.items():
    for page in doc:
        text = page.get_text(sort=True)
        page_rows.append({
            "문서": doc_id,
            "쪽": page.number + 1,
            "글자수": len(text.strip()),
            "숫자비율": digit_ratio(text),
        })
pages = pd.DataFrame(page_rows)

for doc_id in DOCS:
    empty = pages[(pages["문서"] == doc_id) & (pages["글자수"] < 50)]["쪽"].tolist()
    print(doc_id, "글자가 거의 없는 쪽:", empty)

pages.groupby("문서")["글자수"].describe()[["count", "mean", "min", "max"]]

D01 글자가 거의 없는 쪽: [4]
D02 글자가 거의 없는 쪽: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 36, 56, 86, 116, 118, 140, 172, 174, 188, 190, 232, 234, 250, 252, 256, 258, 282]
D03 글자가 거의 없는 쪽: [2, 13]


,count,mean,min,max
문서,,,,
D01,40.0,2305.475000,0.0,4704.0
D02,282.0,1652.216312,0.0,4194.0
D03,133.0,3587.473684,0.0,6233.0


## ④ 쪽 범위 확인
각 쪽의 첫 줄을 보며 문서 구조를 파악하고, 실제로 사용할 쪽 범위를 정합니다.
- D02: 282쪽이라 일부만 사용. 목차 쪽은 본문과 같은 단어가 있어 검색을 방해하므로 제외
- D03: 57쪽부터 통계표인지 확인
- 한 문서에 구간을 여러 개 지정할 수 있게 `(시작쪽, 끝쪽)`의 목록으로 적습니다.

In [17]:
def show_headings(doc_id, start, end):
    """각 쪽의 첫 두 줄을 출력해 문서 구조(목차)를 파악"""
    for page_no in range(start, end + 1):
        lines = [l.strip() for l in docs[doc_id][page_no - 1].get_text(sort=True).splitlines() if l.strip()]
        print(f"{doc_id} p{page_no:>3} | " + " / ".join(lines[:2])[:70])

show_headings("D02", 11, 60)

D02 p 11 | 요약문
D02 p 12 | 
D02 p 13 | 이용자 유의사항 / □ 본 통계는 국가승인통계가 아님
D02 p 14 | 요 약 문 / 들어가는 말
D02 p 15 | 2025년 4분기 콘텐츠산업전년,   동기 대비 매출종사자·   증가수출,   감소 / ▪ 매출액 45조 132억 원전년,  
D02 p 16 | 비중을 차지함그.  다음으로 음악약(  5억 9,519만 달러, 13.0%), 방송 및 영상약(  2억 4,805 / 만 달러
D02 p 17 | 표 2                                ’24년 1분기~’25년 4분기 콘텐츠산업 수출액 규모 / (단
D02 p 18 | 2. 2025년 연간 콘텐츠산업 규모 / 가. 2025년 연간 매출액
D02 p 19 | 표 5                                  ’22년 ~ ’25년 연간 콘텐츠산업 수출액 규모 / (단위
D02 p 20 | 3. 2025년 4분기 콘텐츠산업 생산동향 / 가분기별.    생산지수
D02 p 21 | 4. 2025년 4분기 콘텐츠산업 상장사 현황2) / 가매출.   변화 추이
D02 p 22 | 마전체.   산업3) 대비 콘텐츠산업 상장사 비교(’24년 4분기 및 ’25년 4분기) / ❍ ’25년 4분기 전체 상장사 매
D02 p 23 | 목 차 / 제장1  2025년 4분기 및 연간 콘텐츠산업 규모
D02 p 24 | 4. 콘텐츠산업 생성형 AI 활용 동향 ················································
D02 p 25 | 제장4  2025년 4분기 콘텐츠산업 동향지수 / 1. 지수산출 방법 ·······························
D02 p 26 | 1) 2025년 4분기 모태펀드자펀드/    운영현황 ········································
D02 p 27 | 2) 장르별 상장사 매출 변화 추이 ··················

In [18]:
# D03: 57쪽부터 통계표인지 확인
pages[(pages["문서"] == "D03") & (pages["쪽"].between(50, 62))]

,문서,쪽,글자수,숫자비율
371,D03,50,2566,0.43
372,D03,51,3078,0.51
373,D03,52,2769,0.43
374,D03,53,2985,0.50
375,D03,54,3541,0.51
376,D03,55,3380,0.50
377,D03,56,3205,0.56
378,D03,57,2431,0.04
379,D03,58,1895,0.05
380,D03,59,3999,0.61


In [19]:
# 확인 결과로 정한 사용 범위: 문서마다 (시작쪽, 끝쪽) 구간의 목록
PAGE_RANGES = {
    "D01": [(1, 40)],              # 전체
    "D02": [(13, 22), (37, 54)],   # 요약문 + 1장 (23~34쪽 목차, 35~36쪽 간지 제외)
    "D03": [(1, 56)],              # 요약 + 분야별 설명 (57쪽~ 통계표 제외)
}

def pages_in_range(doc_id):
    """사용 범위에 속한 쪽 번호를 차례로 돌려줌"""
    for start, end in PAGE_RANGES[doc_id]:
        yield from range(start, end + 1)

for doc_id in DOCS:
    print(doc_id, "사용 쪽수:", len(list(pages_in_range(doc_id))))

D01 사용 쪽수: 40
D02 사용 쪽수: 28
D03 사용 쪽수: 56


## ⑤ 글자 추출 품질 비교
같은 쪽을 두 가지 방식으로 추출해 비교합니다. 저장된 그림(`data/explore/`)과 함께 보며 순서가 맞는지 확인합니다.
- 기본 추출: PDF 안에 저장된 순서대로
- `sort=True`: 화면 위치(위→아래, 왼쪽→오른쪽) 순서대로

In [20]:
def save_page(doc_id, page_no, dpi=100):
    """쪽을 그림 파일로 저장 (노트북 화면에는 띄우지 않음)"""
    path = OUT_DIR / f"{doc_id}_p{page_no:03d}.png"
    docs[doc_id][page_no - 1].get_pixmap(dpi=dpi).save(path)
    return path

SAMPLE = ("D02", 15)
print("그림 저장:", save_page(*SAMPLE))
page = docs[SAMPLE[0]][SAMPLE[1] - 1]
print("=== 기본 추출 ===")
print(page.get_text()[:300])
print("=== sort=True (위치순 정렬) ===")
print(page.get_text(sort=True)[:300])

그림 저장: /home/user/git_project/project6/ReportQA/data/explore/D02_p015.png
=== 기본 추출 ===
- xv -
년 분기 콘텐츠산업전년 동기 대비 매출종사자 증가수출 감소
2025
4
, 
·
, 
매출액 
조 
억 원전년 동기 대비 
증가
  
45
132
, 
5.0% 
▪
수출액 
억 
만 달러전년 동기 대비 
감소
  
45
8,825
, 
9.7% 
▪
종사자 
만 
명전년 동기 대비 
증가
  
69
3,090 , 
0.7% 
▪
년 연간 콘텐츠산업전년 대비 매출수출종사자 증가
2025
, 
·
·
  ▪매출액 
조 
억 원전년 대비 
증가
161
4,839
, 
2.6% 
  ▪수출액 
억 
만 달러전년 대비 
증가

=== sort=True (위치순 정렬) ===
     2025년 4분기 콘텐츠산업전년,   동기 대비 매출종사자·   증가수출,   감소

  ▪ 매출액 45조 132억 원전년,   동기 대비 5.0% 증가

  ▪ 수출액 45억 8,825만 달러전년,   동기 대비 9.7% 감소

  ▪ 종사자 69만 3,090명전년,   동기 대비 0.7% 증가


          2025년 연간 콘텐츠산업전년,   대비 매출수출종사자·     ·   증가

  ▪매출액 161조 4,839억 원전년,   대비 2.6% 증가

  ▪수출액 149억 582만 달러전년,   대비 5.9% 증


## ⑥ 표 확인
사용 범위 안에서 PyMuPDF의 표 찾기 기능으로 표를 셉니다. (약 1분 소요)

자동으로 센 숫자는 **후보**입니다. 작은 표나 테두리 상자까지 세기 때문에 실제보다 많을 수 있습니다.

In [21]:
table_rows = []
for doc_id, doc in docs.items():
    for page_no in pages_in_range(doc_id):
        found = doc[page_no - 1].find_tables().tables
        if found:
            table_rows.append({"문서": doc_id, "쪽": page_no, "표개수": len(found)})
tables = pd.DataFrame(table_rows)
tables.groupby("문서")["표개수"].sum()

문서
D01    59
D02    13
D03    59
Name: 표개수, dtype: int64

In [22]:
# 표 하나를 DataFrame으로 바꿔 보기 (쪽 객체를 변수에 담아 둬야 오류가 나지 않음)
example_page = docs["D01"][8 - 1]
example_page.find_tables().tables[0].to_pandas()

,Col0,Col1,Col2,인구,취업자\n고용률,Col5,실업자\n실업률,Col7,비경제활동인구
0,규모,전 체\n청년층1),NaN,"45984\n,","29120 633\n, .",NaN,878 29\n.,NaN,"15986\n,"
1,NaN,NaN,청년층1),"7822\n,","3427\n,",438\n.,266,72\n.,"4128\n,"
2,증감2),전 체\n청년층,NaN,250,40 05\n- - .,NaN,25 01\n.,NaN,264
3,NaN,NaN,청년층,152\n-,255\n-,24\n- .,5,06\n.,98


## ⑦ 차트 후보 찾기
이미지가 있거나 도형(선·사각형)이 많은 쪽을 차트 후보로 봅니다. 쪽 안의 캡션(`< 제목 >`, `그림 1-1`, `표 1-1`)도 함께 보여줍니다.

후보 쪽은 그림으로 저장하니, VS Code에서 열어 **눈으로 확인**해 개수를 확정합니다. (기준: 제목이 붙은 차트 1개 = 1개)

In [23]:
CAPTION = re.compile(r"^\s*(<\s*[^<>]{3,60}?\s*>|\[?\s*(그림|표)\s*\d+(-\d+)?\s*\]?)")

def is_caption(line):
    """< 제목 > 또는 그림 1-1 / 표 1-1 형태의 캡션인지 (숫자만 있는 <100.0> 등은 제외)"""
    m = CAPTION.match(line)
    if not m:
        return False
    inner = re.sub(r"[<>\[\]\s]", "", m.group(1))
    return not re.fullmatch(r"[\d.,%\-()월년전체]+", inner)

chart_rows = []
for doc_id, doc in docs.items():
    for page_no in pages_in_range(doc_id):
        page = doc[page_no - 1]
        lines = [l.strip() for l in page.get_text(sort=True).splitlines() if l.strip()]
        captions = [l for l in lines if is_caption(l)]
        images = len(page.get_images())
        drawings = len(page.get_drawings())
        if images or drawings > 50:
            chart_rows.append({
                "문서": doc_id, "쪽": page_no, "이미지": images, "도형": drawings,
                "캡션": " / ".join(captions)[:80],
            })
charts = pd.DataFrame(chart_rows)
print(charts.groupby("문서").size().rename("차트 후보 쪽수"))
charts.head(15)

문서
D01    33
D02    12
D03    51
Name: 차트 후보 쪽수, dtype: int64


,문서,쪽,이미지,도형,캡션
0,D01,1,6,3,
1,D01,5,7,763,
2,D01,6,1209,2,< 학제별 대졸자의 평균 졸업 소요기간 >
3,D01,7,1,2,< 비경제활동인구의 취업시험 준비 분야 >
4,D01,8,0,65,< 수학상태별 경제활동인구 구성 >
5,D01,9,1,81,< 청년층 경제활동상태 >
6,D01,10,1391,77,< 청년층 수학상태 >
7,D01,11,0,185,< 졸업자의 취업상태 > / < 졸업자의 산업별 취업분포 >
8,D01,12,1139,84,< 졸업자의 직업별 취업분포 >
9,D01,13,0,149,< 청년층 학력별 졸업자수 > / < 대학 졸업 소요기간 >


In [24]:
# 문서별 차트 후보 5쪽씩 그림으로 저장 → VS Code에서 열어 눈으로 확인
for doc_id in DOCS:
    for page_no in charts[charts["문서"] == doc_id]["쪽"].head(5):
        save_page(doc_id, page_no)
print("저장 위치:", OUT_DIR)

저장 위치: /home/user/git_project/project6/ReportQA/data/explore


## ⑧ 결과 요약
문서별 결과를 표 하나로 정리합니다. 이 값을 확인한 뒤 `docs/data_sources.md`에 옮겨 적습니다.

In [25]:
summary = pd.DataFrame({
    "쪽수": {d: docs[d].page_count for d in DOCS},
    "사용 범위": {d: ", ".join(f"{s}~{e}" for s, e in r) for d, r in PAGE_RANGES.items()},
    "사용 쪽수": {d: len(list(pages_in_range(d))) for d in DOCS},
    "표(자동)": tables.groupby("문서")["표개수"].sum(),
    "차트 후보 쪽(자동)": charts.groupby("문서").size(),
})
summary

,쪽수,사용 범위,사용 쪽수,표(자동),차트 후보 쪽(자동)
D01,40,1~40,40,59,33
D02,282,"13~22, 37~54",28,13,12
D03,133,1~56,56,59,51


## 발견한 것

| 문서 | 발견 | 대응 |
|---|---|---|
| D02 | 기본 추출 시 글자 순서가 뒤섞임 ("2025년 4분기" → "년 분기 … 2025 … 4") | `get_text(sort=True)`로 위치순 정렬하면 대부분 해결 |
| D01 | 표를 자동으로 읽으면 숫자가 깨짐 (`2.9` → `29`, `.` 분리) | 3주차 표 구조 반영(v1)에서 해결 방법 검토 |
| D01 | 차트가 작은 이미지 조각 수백~천여 개로 저장됨 | 이미지 추출 대신 쪽 렌더링 후 차트 영역 자르기 (Vision 단계) |
| D02 | 23~34쪽은 목차·표 목차·그림 목차 (본문과 같은 단어 포함) | 13~22, 37~54쪽 두 구간만 사용 |
| D03 | 57쪽부터 통계표 (숫자 위주) | 1~56쪽만 사용 |

> 표·차트 개수는 ⑥, ⑦의 자동 결과와 저장된 그림을 눈으로 확인한 뒤 확정합니다.